# TP4 — Cassandra et PySpark

**Objectif :** lire les données Cassandra avec PySpark, créer un DataFrame, effectuer des transformations et agrégations, puis observer la Lazy Evaluation, les partitions et le Shuffle dans Spark UI.

**Source des données :** table `velib_tp2.stations_by_id`  
**Nombre de lignes attendu :** 20 stations  
**Environnement observé :** Spark `local[4]`, Cassandra accessible sur `127.0.0.1:9042`.

## 1. Initialisation de Spark

In [1]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .appName("TP1-Cassandra-Spark")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
sc = spark.sparkContext
print("Version Spark :", spark.version)
print("Master :", sc.master)
print("Parallélisme :", sc.defaultParallelism)

Version Spark : 4.2.0
Master : local[4]
Parallélisme : 4


## 2. Connexion à Cassandra

In [2]:
%pip install cassandra-driver

Note: you may need to restart the kernel to use updated packages.


In [3]:
from cassandra.cluster import Cluster

In [ ]:
cluster = Cluster(["cass1"])
session = cluster.connect()
print("Connexion Cassandra réussie")

## 3. Vérification du schéma et des données

In [ ]:
keyspace = "velib_tp2"
table = "stations_by_id"
rows = session.execute("SELECT keyspace_name FROM system_schema.keyspaces")
for row in rows:
    print(row.keyspace_name)

## 4. Création du DataFrame Spark

In [ ]:
rows = session.execute("""
    SELECT column_name, type
    FROM system_schema.columns
    WHERE keyspace_name = 'velib_tp2'
      AND table_name = 'stations_by_id'
""")

for row in rows:
    print(f"{row.column_name} : {row.type}")

## 5. Transformations : sélection et filtres

In [ ]:
rows = session.execute("""
    SELECT station_id, name, capacity, capacity_category, zone
    FROM velib_tp2.stations_by_id
    LIMIT 10
""")

for row in rows:
    print(row)

In [ ]:
data = session.execute("""
    SELECT station_id, name, capacity, capacity_category, zone
    FROM velib_tp2.stations_by_id
""")

rows = list(data)

df = spark.createDataFrame(rows)

print("Nombre de lignes :", df.count())
df.printSchema()

In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql import functions as F

# Sélection des colonnes principales
df_selection = df.select(
    "station_id",
    "name",
    "capacity",
    "capacity_category",
    "zone"
)

df_selection.show(10, truncate=False)

In [ ]:
# Stations de petite capacité
df.filter(F.col("capacity") < 30).show(10, truncate=False)

In [ ]:
# Stations de grande capacité
df.filter(F.col("capacity") >= 40).show(10, truncate=False)

## 6. Calculs métier et agrégations

In [ ]:
df_calc = df.withColumn(
    "taux_capacite_pct",
    F.round((F.col("capacity") / 60) * 100, 2)
)

df_calc.select(
    "station_id",
    "name",
    "capacity",
    "taux_capacite_pct"
).show(10, truncate=False)

In [ ]:
df.agg(
    F.avg("capacity").alias("capacite_moyenne"),
    F.min("capacity").alias("capacite_min"),
    F.max("capacity").alias("capacite_max"),
    F.sum("capacity").alias("capacite_totale")
).show()

In [ ]:
df.groupBy("capacity_category").count().show()

In [ ]:
df.groupBy("zone").count().orderBy("zone").show()

In [ ]:
df_niveaux = df.withColumn(
    "niveau_capacite",
    F.when(F.col("capacity") < 30, "faible")
     .when(F.col("capacity") < 45, "moyenne")
     .otherwise("élevée")
)

df_niveaux.select(
    "station_id",
    "name",
    "capacity",
    "niveau_capacite"
).show(20, truncate=False)

In [ ]:
print("Nombre de partitions :", df.rdd.getNumPartitions())

In [ ]:
df_repartition = df.repartition(8)

print("Partitions de df :", df.rdd.getNumPartitions())
print("Partitions de df_repartition :", df_repartition.rdd.getNumPartitions())

## 7. Lazy Evaluation : construire les transformations

In [ ]:
df_lazy = (
    df
    .filter(F.col("capacity") >= 30)
    .withColumn(
        "capacite_doublee",
        F.col("capacity") * 2
    )
    .select(
        "station_id",
        "name",
        "capacity",
        "capacite_doublee"
    )
)

print("Transformation créée.")

## 8. Action : déclencher l'exécution

In [ ]:
df_lazy.show(10, truncate=False)

**Observation :** `show()` est une action. Elle demande à Spark d'exécuter les transformations précédemment définies.

## 9. Plan d'exécution

In [ ]:
df_lazy.explain(True)

**À observer :**
- `Parsed Logical Plan` : plan initial construit à partir des opérations demandées.
- `Analyzed Logical Plan` : plan analysé avec les colonnes et les types.
- `Optimized Logical Plan` : plan logique optimisé.
- `Physical Plan` : stratégie d'exécution retenue.

Pour `filter → withColumn → select`, le plan observé ne contient pas `Exchange` : cette chaîne de transformations ne nécessite pas le Shuffle identifié dans l'agrégation suivante.

## 10. Partitions

In [ ]:
print("Partitions initiales :", df.rdd.getNumPartitions())

df_repartition = df.repartition(8)
print("Partitions après repartition(8) :", df_repartition.rdd.getNumPartitions())

**Résultat observé pendant le TP :** le DataFrame initial possède 4 partitions ; après `repartition(8)`, le nouveau DataFrame en possède 8. `df` reste inchangé.

## 11. Shuffle : agrégation par catégorie

In [ ]:
df_shuffle = df.groupBy("capacity_category").count()
df_shuffle.show()

In [ ]:
df_shuffle.explain(True)

**Élément observé dans le plan physique :**

`Exchange hashpartitioning(capacity_category#17, 4)`

`Exchange` indique ici une redistribution des données (Shuffle) avant l'agrégation. Les identifiants internes (`#17`, par exemple) peuvent varier d'une exécution à l'autre.

In [ ]:
df_shuffle.show()

In [ ]:
print("Partitions initiales :", df.rdd.getNumPartitions())
print("Partitions après repartition(8) :", df.repartition(8).rdd.getNumPartitions())

## 12. Observation de Spark UI : Jobs, Stages et Tasks

Ouvrir **http://localhost:4040** pendant que la session Spark est active.

Observations relevées pendant cette séance :
- Job ID : **25**, statut `SUCCEEDED`, durée affichée **10 s**.
- Stage ID : **30**, durée affichée **10 s**.
- Tasks : **4/4** réussies.
- Shuffle Write du Stage : **568.0 B / 8 records**.
- Par Task : **142 B / 2 records**.
- Locality : `PROCESS_LOCAL`.

Ces valeurs sont les observations de cette exécution ; elles peuvent changer lors d'une nouvelle exécution ou selon la version et l'environnement Spark.

## 13. Conclusion

Ce TP a permis de lire les données de Cassandra dans un DataFrame PySpark, de sélectionner et filtrer des stations, de calculer des indicateurs et de regrouper les stations par catégorie et par zone. La comparaison entre la définition des transformations et l'appel à `show()` illustre la Lazy Evaluation. `repartition(8)` montre la modification du nombre de partitions, tandis que `Exchange hashpartitioning(...)` et les métriques de Spark UI permettent d'observer le Shuffle ainsi que les Jobs, Stages et Tasks.